# Gold Layer Transformation
## FinMark Data Pipeline — Milestone 2

This notebook documents the Gold Layer transformation for the FinMark data pipeline.  
It takes the cleaned Silver Layer datasets and produces 6 analysis-ready output files,  
one for each dashboard and a master KPI overview for the documentation team.

### Input Files (Silver Layer)
| File | Description |
|---|---|
| `event_logs_clean.csv` | Cleaned user event logs (2,000 rows) |
| `marketing_summary_clean.csv` | Daily marketing summary (100 rows) |
| `trend_report_clean.csv` | Weekly sales trend (20 rows) |
| `silver_quality_issues.csv` | Known data quality issues flagged in Silver |
| `silver_validation_report.csv` | Silver layer validation check results |

### Output Files (Gold Layer)
| File | Feeds Dashboard |
|---|---|
| `kpi_master_summary.csv` | Documentation / Overview |
| `funnel_conversion.csv` | Dashboard 1 – Customer Journey & Conversion Funnel |
| `product_feature_by_hour.csv` | Dashboard 2 – Product & Feature Usage |
| `ops_hourly_load.csv` | Dashboard 3 – Operations & System Health |
| `ops_checkout_health.csv` | Dashboard 3 – Operations & System Health |
| `compliance_quality_summary.csv` | Dashboard 4 – Data Privacy, Integrity & Compliance |


## 0. Setup — Load Libraries and Silver Layer Data

In [2]:
import os

# Navigate up one level from /notebooks to the project root
os.chdir(os.path.join(os.getcwd(), ".."))
print("Working directory:", os.getcwd())

Working directory: c:\Users\USER\MS2-finmark-data-pipeline


In [3]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Load Silver Layer outputs
events   = pd.read_csv("data/silver/event_logs_clean.csv", parse_dates=["event_date", "event_timestamp"])
marketing = pd.read_csv("data/silver/marketing_summary_clean.csv", parse_dates=["date"])
trends   = pd.read_csv("data/silver/trend_report_clean.csv")
quality  = pd.read_csv("data/silver/silver_quality_issues.csv")
validation = pd.read_csv("data/silver/silver_validation_report.csv")

print("event_logs_clean    :", events.shape)
print("marketing_summary   :", marketing.shape)
print("trend_report_clean  :", trends.shape)
print("silver_quality_issues:", quality.shape)
print("silver_validation   :", validation.shape)

event_logs_clean    : (2000, 10)
marketing_summary   : (100, 7)
trend_report_clean  : (20, 4)
silver_quality_issues: (5, 4)
silver_validation   : (13, 4)


## 0b. Gold Layer Imputation — Missing Checkout Amounts

**Problem:** 142 of 260 checkout rows (≈54.6%) have no `amount` value. The missing values originate in the source system - `amount` was never recorded for those events. They were flagged in Silver and rows were kept rather than dropping them.

**Decision:** Fill missing `amount` values with the **median** of known checkout amounts.

| Statistic | Value |
|---|---|
| Known checkout rows | 118 |
| Missing checkout rows | 142 |
| Mean (known) | \₱1,597.58 |
| **Median (known)** | **\₱1,686.80** |
| Std Dev | \₱889.15 |

A Shapiro-Wilk normality test rejects normality (p < 0.0001). The distribution is non-normal, so the median is the more robust central tendency measure rather than the mean — it is not pulled by the long lower tail the way the mean is. Using the median also slightly *understates* imputed revenue to keep estimates conservative.

**Transparency:** Imputed rows retain `is_amount_missing = True` and `amount_quality_status = 'imputed'` so downstream dashboards can always distinguish real from estimated amounts.

> **Limitation:** Imputed amounts are statistical estimates. Total revenue figures that include them should be labelled *'includes imputed values'* in dashboards.

In [6]:
# --- Gold Layer Imputation: fill missing checkout amounts with median ---

# Compute median from known (non-missing) checkout amounts only
checkout_median = events.loc[
    (events["is_checkout"] == True) & (events["amount"].notna()),
    "amount"
].median()

print(f"Imputation value (median of known checkout amounts): ${checkout_median:,.2f}")

# Apply imputation: only fill where checkout AND amount is missing
mask = (events["is_checkout"] == True) & (events["amount"].isna())
events.loc[mask, "amount"] = checkout_median

# Update quality status flag for imputed rows
events.loc[mask, "amount_quality_status"] = "imputed"

# Re-derive checkouts view after imputation
checkouts = events[events["is_checkout"] == True].copy()

# Verify
still_missing = checkouts["amount"].isna().sum()
imputed_count = (checkouts["amount_quality_status"] == "imputed").sum()
print(f"Checkout rows still missing amount after imputation: {still_missing}")
print(f"Checkout rows marked 'imputed':                      {imputed_count}")
print(f"\nCheckout amount stats after imputation:")
print(checkouts["amount"].describe().round(2))

Imputation value (median of known checkout amounts): $1,686.80
Checkout rows still missing amount after imputation: 0
Checkout rows marked 'imputed':                      142

Checkout amount stats after imputation:
count     260.00
mean     1646.30
std       599.27
min       125.36
25%      1686.80
50%      1686.80
75%      1686.80
max      2952.07
Name: amount, dtype: float64


## 1. KPI Master Summary
**Output:** `kpi_master_summary.csv`  
**Purpose:** A single-table overview of the most important business metrics across all three source datasets. Used by the documentation writer and as a reference for the dashboard team.

**Calculated fields:**
- Total, average, peak, and minimum revenue from `marketing_summary_clean`
- Average daily active users and total new customers
- Revenue per active user (total_sales / users_active)
- Checkout rate and missing amount rate from `event_logs_clean`
- Average weekly sales growth rate and count of positive-growth weeks from `trend_report_clean`


In [7]:
checkouts = events[events["is_checkout"] == True]

kpi = pd.DataFrame([
    {"metric": "Total Revenue (All Days)",         "value": round(marketing["total_sales"].sum(), 2),                          "unit": "USD",       "source": "marketing_summary"},
    {"metric": "Average Daily Revenue",            "value": round(marketing["total_sales"].mean(), 2),                         "unit": "USD/day",   "source": "marketing_summary"},
    {"metric": "Peak Daily Revenue",               "value": round(marketing["total_sales"].max(), 2),                          "unit": "USD",       "source": "marketing_summary"},
    {"metric": "Lowest Daily Revenue",             "value": round(marketing["total_sales"].min(), 2),                          "unit": "USD",       "source": "marketing_summary"},
    {"metric": "Average Daily Active Users",       "value": round(marketing["users_active"].mean(), 1),                        "unit": "users",     "source": "marketing_summary"},
    {"metric": "Total New Customers",              "value": int(marketing["new_customers"].sum()),                              "unit": "customers", "source": "marketing_summary"},
    {"metric": "Avg Revenue per Active User",      "value": round((marketing["total_sales"] / marketing["users_active"]).mean(), 2), "unit": "USD/user", "source": "marketing_summary"},
    {"metric": "Total Events Logged",              "value": len(events),                                                        "unit": "events",    "source": "event_logs"},
    {"metric": "Total Checkout Events",            "value": int(events["is_checkout"].sum()),                                   "unit": "events",    "source": "event_logs"},
    {"metric": "Checkout Rate",                    "value": round(events["is_checkout"].mean() * 100, 2),                       "unit": "%",         "source": "event_logs"},
    {"metric": "Missing Amount Rate (Checkouts)",  "value": round(checkouts["is_amount_missing"].mean() * 100, 2),              "unit": "%",         "source": "event_logs"},
    {"metric": "Average Weekly Sales Growth Rate", "value": round(trends["sales_growth_rate"].mean() * 100, 2),                 "unit": "%",         "source": "trend_report"},
    {"metric": "Weeks with Positive Growth",       "value": int((trends["sales_growth_rate"] > 0).sum()),                       "unit": "weeks",     "source": "trend_report"},
])

kpi.to_csv("data/gold/kpi_master_summary.csv", index=False)
print("Saved: kpi_master_summary.csv")
kpi

Saved: kpi_master_summary.csv


,metric,value,unit,source
0,Total Revenue (All Days),5767580.45,USD,marketing_summary
1,Average Daily Revenue,57675.80,USD/day,marketing_summary
2,Peak Daily Revenue,89585.41,USD,marketing_summary
3,Lowest Daily Revenue,20780.11,USD,marketing_summary
4,Average Daily Active Users,273.30,users,marketing_summary
5,Total New Customers,765.00,customers,marketing_summary
6,Avg Revenue per Active User,281.11,USD/user,marketing_summary
7,Total Events Logged,2000.00,events,event_logs
8,Total Checkout Events,260.00,events,event_logs
9,Checkout Rate,13.00,%,event_logs


## 2. Funnel Conversion Table
**Output:** `funnel_conversion.csv`  
**Feeds:** Dashboard 1 — Customer Journey & Conversion Funnel  
**Purpose:** Shows how many users reached each stage of the FinMark user journey, the drop-off between stages, and the conversion rate relative to the top of the funnel.

**Funnel stages mapped from event types:**

| Stage | Event Type |
|---|---|
| Stage 1 | page_view |
| Stage 2 | search |
| Stage 3 | add_to_cart |
| Stage 4 | wishlist_add |
| Stage 5 | checkout |

**Note:** Because the Silver dataset is a snapshot with no session IDs, we cannot trace individual user paths. Counts represent total occurrences per event type, not unique sessions. This is a known data limitation documented in Milestone 1.


In [8]:
funnel_map = {
    "page_view":   ("Stage 1", "Product View"),
    "search":      ("Stage 2", "Search"),
    "add_to_cart": ("Stage 3", "Add to Cart"),
    "wishlist_add":("Stage 4", "Wishlist Add"),
    "checkout":    ("Stage 5", "Checkout"),
}

funnel_rows = []
for etype, (stage_num, stage_name) in funnel_map.items():
    count = len(events[events["event_type"] == etype])
    funnel_rows.append({
        "stage_order": stage_num,
        "stage_name": stage_name,
        "event_type": etype,
        "event_count": count,
    })

funnel_df = pd.DataFrame(funnel_rows).sort_values("stage_order").reset_index(drop=True)
top = funnel_df["event_count"].iloc[0]
funnel_df["drop_off_from_previous"] = funnel_df["event_count"].diff().fillna(0).apply(lambda x: max(0, -x)).astype(int)
funnel_df["conversion_rate_pct"] = (funnel_df["event_count"] / top * 100).round(1)

funnel_df.to_csv("data/gold/funnel_conversion.csv", index=False)
print("Saved: funnel_conversion.csv")
funnel_df

Saved: funnel_conversion.csv


,stage_order,stage_name,event_type,event_count,drop_off_from_previous,conversion_rate_pct
0,Stage 1,Product View,page_view,233,0,100.0
1,Stage 2,Search,search,251,0,107.7
2,Stage 3,Add to Cart,add_to_cart,252,0,108.2
3,Stage 4,Wishlist Add,wishlist_add,260,0,111.6
4,Stage 5,Checkout,checkout,260,0,111.6


## 3. Product Feature Usage by Hour
**Output:** `product_feature_by_hour.csv`  
**Feeds:** Dashboard 2 — Product & Feature Usage  
**Purpose:** Shows how often each platform feature (event type) was used at each hour of the day. This allows the product team to identify peak usage windows per feature, which is directly relevant to FinMark's scale-up goal (500 → 3,000 orders/day).

**Transformation:** Group `event_logs_clean` by `event_hour` and `event_type`, count occurrences, then pivot so each event type becomes a column.


In [9]:
hourly = events.groupby(["event_hour", "event_type"]).size().reset_index(name="event_count")

feature_by_hour = hourly.pivot_table(
    index="event_hour",
    columns="event_type",
    values="event_count",
    fill_value=0
).reset_index()
feature_by_hour.columns.name = None

feature_by_hour.to_csv("data/gold/product_feature_by_hour.csv", index=False)
print("Saved: product_feature_by_hour.csv")
print(f"Shape: {feature_by_hour.shape} — {len(feature_by_hour)} hours x {len(feature_by_hour.columns)} columns")
feature_by_hour

Saved: product_feature_by_hour.csv
Shape: (24, 9) — 24 hours x 9 columns


,event_hour,add_to_cart,checkout,login,logout,page_view,profile_update,search,wishlist_add
0,0,6.0,6.0,9.0,8.0,10.0,7.0,9.0,12.0
1,1,5.0,8.0,9.0,5.0,9.0,11.0,4.0,7.0
2,2,13.0,10.0,16.0,10.0,11.0,18.0,13.0,6.0
3,3,11.0,16.0,10.0,4.0,6.0,9.0,13.0,9.0
4,4,10.0,10.0,10.0,8.0,9.0,5.0,6.0,8.0
5,5,9.0,16.0,10.0,7.0,11.0,10.0,11.0,11.0
6,6,9.0,10.0,13.0,6.0,10.0,13.0,5.0,9.0
7,7,9.0,9.0,12.0,7.0,8.0,9.0,10.0,10.0
8,8,10.0,10.0,19.0,9.0,9.0,13.0,13.0,22.0
9,9,12.0,10.0,14.0,7.0,8.0,10.0,14.0,11.0


## 4. Operations — Hourly System Load
**Output:** `ops_hourly_load.csv`  
**Feeds:** Dashboard 3 — Operations & System Health  
**Purpose:** Provides a proxy for system load at each hour of the day using event volume. This addresses the limitation identified in Milestone 1: FinMark has no real-time order counter but needs to prepare for scaling to 3,000 orders/day.

**Calculated fields:**
- `total_events` — total activity volume per hour
- `checkout_events` — number of checkout events (highest-load transactions)
- `unique_users` — distinct active users per hour
- `checkout_rate_pct` — checkout share of total events
- `load_category` — Low / Medium / High / Peak based on event volume thresholds


In [10]:
ops_hourly = events.groupby("event_hour").agg(
    total_events=("event_type", "count"),
    checkout_events=("is_checkout", "sum"),
    unique_users=("user_id", "nunique"),
).reset_index()

ops_hourly["checkout_rate_pct"] = (ops_hourly["checkout_events"] / ops_hourly["total_events"] * 100).round(2)
ops_hourly["load_category"] = pd.cut(
    ops_hourly["total_events"],
    bins=[0, 60, 85, 100, 999],
    labels=["Low", "Medium", "High", "Peak"]
)

ops_hourly.to_csv("data/gold/ops_hourly_load.csv", index=False)
print("Saved: ops_hourly_load.csv")
print("\nLoad distribution:")
print(ops_hourly["load_category"].value_counts().sort_index())
ops_hourly

Saved: ops_hourly_load.csv

Load distribution:
load_category
Low        1
Medium    12
High       9
Peak       2
Name: count, dtype: int64


,event_hour,total_events,checkout_events,unique_users,checkout_rate_pct,load_category
0,0,67,6,64,8.96,Medium
1,1,58,8,55,13.79,Low
2,2,97,10,88,10.31,High
3,3,78,16,73,20.51,Medium
4,4,66,10,60,15.15,Medium
5,5,85,16,75,18.82,Medium
6,6,75,10,70,13.33,Medium
7,7,74,9,67,12.16,Medium
8,8,105,10,89,9.52,Peak
9,9,86,10,83,11.63,High


## 5. Operations — Daily Checkout Health
**Output:** `ops_checkout_health.csv`  
**Feeds:** Dashboard 3 — Operations & System Health  
**Purpose:** Summarizes checkout performance per day, flagging days where the missing amount rate is high. This is a carryover data quality issue from the Silver Layer: 142 out of 260 checkout rows have no `amount` value, meaning captured revenue is understated.

**Status flags:**
- `Normal` — missing amount rate below 50%
- `Warning` — missing amount rate 50–70%
- `Critical` — missing amount rate above 70%


In [11]:
checkout_daily = checkouts.groupby("event_date").agg(
    total_checkouts=("event_type", "count"),
    missing_amount=("is_amount_missing", "sum"),
    captured_revenue=("amount", "sum"),
).reset_index()

checkout_daily["failed_amount_rate_pct"] = (
    checkout_daily["missing_amount"] / checkout_daily["total_checkouts"] * 100
).round(2)
checkout_daily["captured_revenue"] = checkout_daily["captured_revenue"].round(2)
checkout_daily["status"] = checkout_daily["failed_amount_rate_pct"].apply(
    lambda x: "Critical" if x > 70 else ("Warning" if x > 50 else "Normal")
)

checkout_daily.to_csv("data/gold/ops_checkout_health.csv", index=False)
print("Saved: ops_checkout_health.csv")
print("\nStatus breakdown:")
print(checkout_daily["status"].value_counts())
checkout_daily

Saved: ops_checkout_health.csv

Status breakdown:
status
Warning    5
Normal     1
Name: count, dtype: int64


,event_date,total_checkouts,missing_amount,captured_revenue,failed_amount_rate_pct,status
0,2023-06-01,38,23,63350.36,60.53,Warning
1,2023-06-02,46,29,80032.84,63.04,Warning
2,2023-06-03,52,27,83377.38,51.92,Warning
3,2023-06-04,43,22,71312.14,51.16,Warning
4,2023-06-05,39,21,63910.06,53.85,Warning
5,2023-06-06,42,20,66056.35,47.62,Normal


## 6. Data Quality & Compliance Summary
**Output:** `compliance_quality_summary.csv`  
**Feeds:** Dashboard 4 — Data Privacy, Integrity & Compliance  
**Purpose:** Aggregates data quality and privacy metrics from the Silver Layer validation outputs. This gives the compliance dashboard its source numbers without exposing raw records.

**Sources used:**
- `silver_quality_issues.csv` — known issues flagged during Silver cleaning
- `silver_validation_report.csv` — pass/fail results of Silver validation checks
- Direct counts from `event_logs_clean`


In [12]:
total_events = len(events)
missing_amounts = events["amount"].isna().sum()
checkout_count = int(events["is_checkout"].sum())
missing_checkout = int(checkouts["is_amount_missing"].sum())
duplicate_count = int(quality[quality["issue_type"].str.contains("duplicate", case=False)]["count"].values[0])
flagged_non_checkout = int(quality[quality["issue_type"].str.contains("Non-checkout", case=False)]["count"].values[0])
validation_passed = int((validation["status"] == "PASS").sum())
validation_failed = int((validation["status"] == "FAIL").sum())

compliance = pd.DataFrame([
    {"metric": "Total Silver event records",              "value": total_events,                        "status": "INFO"},
    {"metric": "Duplicate rows removed",                  "value": duplicate_count,                     "status": "PASS"},
    {"metric": "Silver validation checks passed",         "value": validation_passed,                   "status": "PASS"},
    {"metric": "Silver validation checks failed",         "value": validation_failed,                   "status": "INFO"},
    {"metric": "Missing amount fields (all events)",      "value": int(missing_amounts),                "status": "WARNING"},
    {"metric": "Missing amount rate — all events (%)",   "value": round(missing_amounts/total_events*100, 2), "status": "WARNING"},
    {"metric": "Checkout rows with missing amount",       "value": missing_checkout,                    "status": "WARNING"},
    {"metric": "Missing checkout amount rate (%)",        "value": round(missing_checkout/checkout_count*100, 2), "status": "WARNING"},
    {"metric": "Non-checkout rows with amount (flagged)", "value": flagged_non_checkout,                "status": "WARNING"},
    {"metric": "User IDs anonymized",                     "value": "Yes — U-prefixed pseudonymous IDs", "status": "PASS"},
    {"metric": "PII fields detected in dataset",          "value": "None",                              "status": "PASS"},
])

compliance.to_csv("data/gold/compliance_quality_summary.csv", index=False)
print("Saved: compliance_quality_summary.csv")
compliance

Saved: compliance_quality_summary.csv


,metric,value,status
0,Total Silver event records,2000,INFO
1,Duplicate rows removed,0,PASS
2,Silver validation checks passed,13,PASS
3,Silver validation checks failed,0,INFO
4,Missing amount fields (all events),874,WARNING
5,Missing amount rate — all events (%),43.7,WARNING
6,Checkout rows with missing amount,142,WARNING
7,Missing checkout amount rate (%),54.62,WARNING
8,Non-checkout rows with amount (flagged),866,WARNING
9,User IDs anonymized,Yes — U-prefixed pseudonymous IDs,PASS


## 7. Gold Layer Output Summary

| File | Rows | Feeds | Key Fields |
|---|---|---|---|
| `kpi_master_summary.csv` | 13 | Documentation | metric, value, unit, source |
| `funnel_conversion.csv` | 5 | Dashboard 1 | stage_name, event_count, conversion_rate_pct |
| `product_feature_by_hour.csv` | 24 | Dashboard 2 | event_hour, [one col per event type] |
| `ops_hourly_load.csv` | 24 | Dashboard 3 | event_hour, total_events, load_category |
| `ops_checkout_health.csv` | 6 | Dashboard 3 | event_date, captured_revenue, status |
| `compliance_quality_summary.csv` | 11 | Dashboard 4 | metric, value, status |

### Next Steps
The outputted 6 CSVs will be directly uploaded into **Tableau Public** or **Power BI** as flat file data sources to generate the required dashboards for business insights.
